## Exceedance probability for total precipitation compared to 2026.  

In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import scipy as pi


df = pd.read_csv('Data/mores_data.csv')
df.head()


,Date,SWE (in),Snow Depth (in),Avg Air Temp (deg F),Avg Relative Humidity (%),Accumulated Precip (in),Annual Precip (in),Annual Precip Exceedance (%),Year
0,1980-11-03,NaN,NaN,NaN,NaN,1.3,1.3,NaN,1981
1,1980-11-04,NaN,NaN,NaN,NaN,NaN,1.3,NaN,1981
2,1980-11-05,NaN,NaN,NaN,NaN,NaN,1.3,NaN,1981
3,1980-11-06,NaN,NaN,NaN,NaN,NaN,1.3,NaN,1981
4,1980-11-07,NaN,NaN,NaN,NaN,NaN,1.3,NaN,1981


In [2]:
df = pd.read_csv("Data/mores_data.csv")
df["Date"] = pd.to_datetime(df["Date"])


df.to_csv("Data/mores_data.csv", index=False)

- Use a Weibull plotting position to estimate the exceedance probability for each annual precipitation total:Rank the largest annual total first (rank 1). Since your annual total is repeated on every daily row, calculate ranks on one row per year, not on the daily data.

In [3]:

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Water year ends in the named year: Oct 1980–Sep 1981 is 1981
df["Year"] = df["Date"].dt.year + (df["Date"].dt.month >= 10)

precip_col = "Accumulated Precip (in)"

# Difference consecutive recorded cumulative readings. If the gauge resets,
# count the new reading as precipitation since the reset.
observations = df.loc[df[precip_col].notna(), ["Year", precip_col]].copy()
change = observations[precip_col].diff()
observations["Precipitation Increment (in)"] = change.where(
    change >= 0, observations[precip_col]
).fillna(observations[precip_col])

# Sum increments by water year
annual_totals = observations.groupby("Year")["Precipitation Increment (in)"].sum()

# Exclude partial years from the probability ranking
complete_annual = annual_totals.loc[
    ~annual_totals.index.isin([1981, 2027])
].to_frame(name="Annual Precip (in)")

# Rank wettest first; average ranks handle ties
n = len(complete_annual)
complete_annual["Rank"] = complete_annual["Annual Precip (in)"].rank(
    ascending=False, method="average"
)
complete_annual["Exceedance Probability (%)"] = (
    complete_annual["Rank"] / (n + 1) * 100
)

# Add the annual total and probability to every daily row in that water year
df["Annual Precip (in)"] = df["Year"].map(annual_totals).round(1)
df["Annual Precip Exceedance (%)"] = df["Year"].map(
    complete_annual["Exceedance Probability (%)"]
)

display(complete_annual.sort_values("Annual Precip (in)", ascending=False))

,Annual Precip (in),Rank,Exceedance Probability (%)
Year,,,
1997,68.3,1.0,2.173913
2017,65.2,2.0,4.347826
1982,61.4,3.0,6.521739
1996,60.7,4.0,8.695652
1983,59.0,5.0,10.869565
1986,58.7,6.0,13.043478
2011,56.6,7.0,15.217391
2006,56.5,8.0,17.391304
1995,56.1,9.0,19.565217


### Exceedance probability for stream flow volume compared to 2026. 

In [4]:
df2 = pd.read_csv('Data/Twinsprings_alltime.csv')
df2.head()

/var/folders/9n/npt_hczs5j115291dbmz7jnh0000gn/T/ipykernel_96291/1084890759.py:1: DtypeWarning: Columns (3,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv('Data/Twinsprings_alltime.csv')


,id,approval_status,last_modified,method_category,monitoring_location_id,parameter_code,qualifier,statistic_id,time,time_series_id,unit_of_measure,value,x,y,Annual Streamflow Depth (mm),Annual Streamflow Volume (m3)
0,65073363-2d0f-483a-8783-3e3c2205e93d,Approved,2025-08-13T15:57:55.015543+00:00,NaN,USGS-13185000,60,NaN,11,1986-10-08T06:00:00+00:00,1bac65d123da4ae08ebf1b12ecaf0d95,ft^3/s,604.0,-115.725278,43.668056,33.335932,7.183560e+07
1,12a871d8-d5d0-47db-9bb0-95ebaa85d9d9,Approved,2025-08-13T15:57:55.015543+00:00,NaN,USGS-13185000,60,NaN,11,1986-10-08T06:15:00+00:00,1bac65d123da4ae08ebf1b12ecaf0d95,ft^3/s,648.0,-115.725278,43.668056,33.335932,7.183560e+07
2,347b8fca-7d81-45bf-9213-1b16033b2e6b,Approved,2025-08-13T15:57:55.015543+00:00,NaN,USGS-13185000,60,NaN,11,1986-10-08T06:30:00+00:00,1bac65d123da4ae08ebf1b12ecaf0d95,ft^3/s,604.0,-115.725278,43.668056,33.335932,7.183560e+07
3,25aaa074-ad6e-4780-a34e-0dcb4d1488e0,Approved,2025-08-13T15:57:55.015543+00:00,NaN,USGS-13185000,60,NaN,11,1986-10-08T06:45:00+00:00,1bac65d123da4ae08ebf1b12ecaf0d95,ft^3/s,598.0,-115.725278,43.668056,33.335932,7.183560e+07
4,a11ab19e-dff6-4883-ac96-291b6f08926a,Approved,2025-08-13T15:57:55.015543+00:00,NaN,USGS-13185000,60,NaN,11,1986-10-08T07:00:00+00:00,1bac65d123da4ae08ebf1b12ecaf0d95,ft^3/s,598.0,-115.725278,43.668056,33.335932,7.183560e+07


- Annual Streamflow Depth (mm), Annual Streamflow Volume (m3)
The values use discharge records in ft^3/s, integrate consecutive 15-minute readings, and convert the resulting annual volume to basin-average depth using the 2,154.9 km² watershed area. Each year’s totals are repeated on the rows for that year. The watershed area is found by converting the watershed area 832 square miles from USGS gauge website to km2. 

In [5]:
path = 'Data/Twinsprings_alltime.csv'
watershed_area_km2 = 2154.9

# Read raw discharge measurements only (not the ft gage-height records)
flow = pd.read_csv(
    path,
    usecols=["time", "unit_of_measure", "value"]
)
flow = flow.loc[flow["unit_of_measure"] == "ft^3/s"].copy()

# Convert UTC timestamps to local Twin Springs time
flow["time"] = pd.to_datetime(flow["time"], utc=True).dt.tz_convert(
    "America/Boise"
)
flow["value"] = pd.to_numeric(flow["value"], errors="coerce")
flow = (
    flow.dropna(subset=["time", "value"])
        .sort_values("time")
        .reset_index(drop=True)
)

# Integrate each consecutive 15-minute interval using mean discharge
interval = flow["time"].shift(-1) - flow["time"]
next_flow = flow["value"].shift(-1)
valid = interval.eq(pd.Timedelta(minutes=15)) & next_flow.notna()

midpoint = flow["time"] + interval / 2
interval_volume_m3 = (
    (flow["value"] + next_flow) / 2
    * interval.dt.total_seconds()
    * 0.028316846592  # cubic feet to cubic meters
)

intervals = pd.DataFrame({
    "Water Year": midpoint.dt.year
                  + (midpoint.dt.month >= 10).astype(int),
    "Volume (m3)": interval_volume_m3,
}).loc[valid]

annual_flow = (
    intervals.groupby("Water Year", as_index=False)["Volume (m3)"]
             .sum()
)

# Convert annual volume to basin-average depth
annual_flow["Depth (mm)"] = (
    annual_flow["Volume (m3)"]
    / (watershed_area_km2 * 1_000_000)
    * 1_000
)

# Omit the incomplete first 
annual_flow = annual_flow.loc[
    ~annual_flow["Water Year"].isin([1987])
].copy()

annual_flow["Rank"] = annual_flow["Depth (mm)"].rank(
    ascending=False,
    method="min"
).astype(int)


n = len(annual_flow)
if n == 0:
    raise ValueError("No complete water years are available to rank.")

annual_flow["Depth Exceedance Probability (%)"] = (
    annual_flow["Depth (mm)"].rank(ascending=False, method="average")
    / (n + 1) * 100
)
annual_flow["Volume Exceedance Probability (%)"] = (
    annual_flow["Volume (m3)"].rank(ascending=False, method="average")
    / (n + 1) * 100
)

annual_flow = annual_flow.sort_values("Volume (m3)", ascending=False)
display(annual_flow)

/var/folders/9n/npt_hczs5j115291dbmz7jnh0000gn/T/ipykernel_96291/61875424.py:25: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  valid = interval.eq(pd.Timedelta(minutes=15)) & next_flow.notna()


,Water Year,Volume (m3),Depth (mm),Rank,Depth Exceedance Probability (%),Volume Exceedance Probability (%)
30,2017.0,1.921915e+09,891.881277,1,2.5,2.5
10,1997.0,1.810946e+09,840.385231,2,5.0,5.0
9,1996.0,1.482007e+09,687.738223,3,7.5,7.5
24,2011.0,1.435013e+09,665.930291,4,10.0,10.0
19,2006.0,1.330251e+09,617.314308,5,12.5,12.5
25,2012.0,1.291907e+09,599.520635,6,15.0,15.0
32,2019.0,1.176816e+09,546.111458,7,17.5,17.5
36,2023.0,1.137117e+09,527.688928,8,20.0,20.0
11,1998.0,1.125771e+09,522.423834,9,22.5,22.5
31,2018.0,1.112645e+09,516.332669,10,25.0,25.0


### Exceedance value SWE. 

In [8]:
mores = pd.read_csv("Data/mores_data.csv", parse_dates=["Date"])

april1_swe = mores.loc[
    (mores["Date"].dt.month == 4) & (mores["Date"].dt.day == 1),
    ["Year", "Date", "SWE (in)"],
].dropna(subset=["SWE (in)"]).copy()

if april1_swe["Year"].duplicated().any():
    raise ValueError("Expected one April 1 SWE observation per water year.")

n = len(april1_swe)
if n == 0:
    raise ValueError("No April 1 SWE observations are available to rank.")

april1_swe["Rank"] = april1_swe["SWE (in)"].rank(
    ascending=False, method="average"
)
april1_swe["SWE Exceedance Probability (%)"] = (
    april1_swe["Rank"] / (n + 1) * 100
)

annual_swe = april1_swe.sort_values("Rank").reset_index(drop=True)
display(annual_swe)

,Year,Date,SWE (in),Rank,SWE Exceedance Probability (%)
0,1982,1982-04-01,49.3,1.0,2.222222
1,1983,1983-04-01,47.8,2.0,4.444444
2,1986,1986-04-01,47.2,3.0,6.666667
3,1997,1997-04-01,43.3,4.0,8.888889
4,1999,1999-04-01,41.5,5.0,11.111111
5,1993,1993-04-01,38.7,6.0,13.333333
6,1995,1995-04-01,38.4,7.0,15.555556
7,1984,1984-04-01,38.0,8.0,17.777778
8,2006,2006-04-01,36.6,9.0,20.000000
9,1989,1989-04-01,36.5,10.0,22.222222
